# MÓDULO 01 — Inventario de escenas Landsat 8/9

**Proyecto:** `lst-barranquilla-suhi-deep-learning`  
**Área:** Barranquilla, Colombia  
**Periodo:** 2013–2026  
**WRS:** Path 9 / Row 52  
**Colecciones GEE:**
- `LANDSAT/LC08/C02/T1_L2`
- `LANDSAT/LC09/C02/T1_L2`

## Objetivo

Construir el inventario tabular de escenas Landsat 8/9 Collection 2 Level-2 disponibles para el path/row de Barranquilla durante el periodo de análisis.

Este módulo no descarga imágenes, no modifica productos raster y no calcula índices espectrales. Su función es documentar la disponibilidad temporal de escenas y generar tablas CSV trazables para los módulos posteriores.

## Entradas

- Google Earth Engine.
- Landsat 8 Collection 2 Level-2.
- Landsat 9 Collection 2 Level-2.
- `WRS_PATH = 9`.
- `WRS_ROW = 52`.

## Salidas

Las salidas se guardan en:

```text
data/scene_inventory/
```

Archivos generados:

```text
scene_inventory_landsat89.csv
annual_scene_count_landsat89.csv
monthly_scene_count_landsat89.csv
```

## Nota de reproducibilidad

El identificador del proyecto de Earth Engine no se fija dentro del notebook. Cada usuario debe configurar su propio proyecto si su entorno lo requiere.


In [1]:
# ============================================================
# CELDA 1 — Instalación opcional e imports
# ============================================================

import sys
import subprocess
from pathlib import Path

# En Google Colab, earthengine-api suele estar disponible.
# Si no lo está, se instala de forma silenciosa.
try:
    import ee
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "earthengine-api",
    ])
    import ee

import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)


IN_COLAB: True


In [2]:
# ============================================================
# CELDA 2 — Directorio del repositorio y salidas
# ============================================================

# Este notebook está diseñado para ejecutarse desde la raíz del repositorio
# o desde la carpeta notebooks/.
#
# Si se ejecuta en Colab, puede montar Google Drive, pero las rutas personales
# no quedan fijadas en el código.

if IN_COLAB:
    drive.mount("/content/drive")

CURRENT_DIR = Path.cwd()

# Si el notebook se ejecuta desde /notebooks, subimos un nivel.
if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
else:
    REPO_DIR = CURRENT_DIR

OUT_DIR = REPO_DIR / "data" / "scene_inventory"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SCENE_CSV = OUT_DIR / "scene_inventory_landsat89.csv"
OUT_YEAR_CSV = OUT_DIR / "annual_scene_count_landsat89.csv"
OUT_MONTH_CSV = OUT_DIR / "monthly_scene_count_landsat89.csv"

print("CURRENT_DIR:", CURRENT_DIR)
print("REPO_DIR:", REPO_DIR)
print("OUT_DIR:", OUT_DIR)


Mounted at /content/drive
CURRENT_DIR: /content
REPO_DIR: /content
OUT_DIR: /content/data/scene_inventory


In [3]:
# ============================================================
# CELDA 3 — Inicialización de Google Earth Engine
# ============================================================

# Configure aquí su proyecto de Earth Engine si su cuenta lo requiere.
# Ejemplo:
# EE_PROJECT_ID = "your-earth-engine-project-id"
#
# Si no se requiere project ID explícito, deje:
EE_PROJECT_ID = None

try:
    if EE_PROJECT_ID:
        ee.Initialize(project=EE_PROJECT_ID)
    else:
        ee.Initialize()
    print("Earth Engine ya estaba inicializado.")
except Exception as e:
    print("Inicialización falló:")
    print(e)
    print("\nAutenticando Earth Engine en modo notebook...")

    ee.Authenticate(auth_mode="notebook")

    if EE_PROJECT_ID:
        ee.Initialize(project=EE_PROJECT_ID)
    else:
        ee.Initialize()

    print("Earth Engine inicializado correctamente.")

print(ee.String("Earth Engine OK").getInfo())


Inicialización falló:
Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

Autenticando Earth Engine en modo notebook...
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=37p1dFQP4haHGsO3wABnPFuEnaFyDWoh9IaqefHEdXM&tc=-xGl-0NvV8yn5OdzVa5c8Ddu1O0YhPK6ZRKg2UeUjc8&cc=Zn4FodqnJDADHy7HDx_RUhXRM5epAN_4p6o-Lut4v_w

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPzz44CRAKZQtAVJn2pgUdCU-1IotNt7PQb6_YU

In [4]:
# ============================================================
# CELDA 4 — Configuración del inventario Landsat 8/9
# ============================================================

START_YEAR = 2013
END_YEAR = 2026

START_DATE = f"{START_YEAR}-01-01"
END_DATE = f"{END_YEAR + 1}-01-01"

WRS_PATH = 9
WRS_ROW = 52

LANDSAT_COLLECTIONS = {
    "LC08": "LANDSAT/LC08/C02/T1_L2",
    "LC09": "LANDSAT/LC09/C02/T1_L2",
}

print("Periodo:", START_DATE, "a", END_DATE)
print("WRS_PATH:", WRS_PATH)
print("WRS_ROW:", WRS_ROW)
print("Colecciones:", LANDSAT_COLLECTIONS)


Periodo: 2013-01-01 a 2027-01-01
WRS_PATH: 9
WRS_ROW: 52
Colecciones: {'LC08': 'LANDSAT/LC08/C02/T1_L2', 'LC09': 'LANDSAT/LC09/C02/T1_L2'}


In [5]:
# ============================================================
# CELDA 5 — Cargar colecciones Landsat 8/9 C2 L2 desde GEE
# ============================================================

def prepare_collection(collection_id: str, sensor_name: str) -> ee.ImageCollection:
    """Filtra una colección Landsat por fecha y WRS path/row."""
    return (
        ee.ImageCollection(collection_id)
        .filterDate(START_DATE, END_DATE)
        .filterMetadata('WRS_PATH', 'equals', WRS_PATH)
        .filterMetadata('WRS_ROW', 'equals', WRS_ROW)
        .map(lambda img: img.set('SENSOR', sensor_name))
    )

l8 = prepare_collection(LANDSAT_COLLECTIONS['LC08'], 'LC08')
l9 = prepare_collection(LANDSAT_COLLECTIONS['LC09'], 'LC09')

l89 = l8.merge(l9).sort('system:time_start')

n_scenes = int(l89.size().getInfo())
print('Total escenas encontradas:', n_scenes)

if n_scenes == 0:
    raise RuntimeError('No se encontraron escenas para la configuración indicada.')


Total escenas encontradas: 382


In [6]:
# ============================================================
# CELDA 6 — Extraer inventario de escenas
# ============================================================

def image_to_feature(img: ee.Image) -> ee.Feature:
    """Convierte metadatos relevantes de una imagen Landsat en Feature."""
    date = ee.Date(img.get('system:time_start'))

    return ee.Feature(None, {
        'scene_id': img.get('LANDSAT_PRODUCT_ID'),
        'sensor': img.get('SENSOR'),
        'date': date.format('YYYY-MM-dd'),
        'year': date.get('year'),
        'month': date.get('month'),
        'doy': date.getRelative('day', 'year').add(1),
        'wrs_path': img.get('WRS_PATH'),
        'wrs_row': img.get('WRS_ROW'),
        'cloud_cover': img.get('CLOUD_COVER'),
        'cloud_cover_land': img.get('CLOUD_COVER_LAND'),
        'collection_category': img.get('COLLECTION_CATEGORY'),
        'processing_level': img.get('PROCESSING_LEVEL'),
        'spacecraft_id': img.get('SPACECRAFT_ID'),
    })

features = l89.map(image_to_feature)
inventory = features.getInfo()['features']
rows = [f['properties'] for f in inventory]

df = pd.DataFrame(rows)

cols = [
    'scene_id',
    'sensor',
    'date',
    'year',
    'month',
    'doy',
    'wrs_path',
    'wrs_row',
    'cloud_cover',
    'cloud_cover_land',
    'collection_category',
    'processing_level',
    'spacecraft_id',
]

df = df[cols].sort_values(['year', 'date', 'sensor']).reset_index(drop=True)

df.to_csv(OUT_SCENE_CSV, index=False)

print('Inventario guardado en:')
print(OUT_SCENE_CSV)
print('Número de escenas:', len(df))

display(df.head(20))
display(df.tail(20))


Inventario guardado en:
/content/data/scene_inventory/scene_inventory_landsat89.csv
Número de escenas: 382


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
0,LC08_L2SP_009052_20130401_20200913_02_T1,LC08,2013-04-01,2013,4,91,9,52,8.20,21.50,T1,L2SP,LANDSAT_8
1,LC08_L2SP_009052_20130513_20200912_02_T1,LC08,2013-05-13,2013,5,133,9,52,16.20,24.11,T1,L2SP,LANDSAT_8
2,LC08_L2SP_009052_20130614_20200912_02_T1,LC08,2013-06-14,2013,6,165,9,52,17.20,15.33,T1,L2SP,LANDSAT_8
3,LC08_L2SP_009052_20130630_20200912_02_T1,LC08,2013-06-30,2013,6,181,9,52,64.25,15.68,T1,L2SP,LANDSAT_8
4,LC08_L2SP_009052_20130716_20200912_02_T1,LC08,2013-07-16,2013,7,197,9,52,73.55,52.27,T1,L2SP,LANDSAT_8
5,LC08_L2SP_009052_20130801_20200912_02_T1,LC08,2013-08-01,2013,8,213,9,52,57.53,26.26,T1,L2SP,LANDSAT_8
6,LC08_L2SP_009052_20130817_20200913_02_T1,LC08,2013-08-17,2013,8,229,9,52,21.81,32.07,T1,L2SP,LANDSAT_8
7,LC08_L2SP_009052_20130902_20200912_02_T1,LC08,2013-09-02,2013,9,245,9,52,11.01,27.56,T1,L2SP,LANDSAT_8
8,LC08_L2SP_009052_20130918_20200912_02_T1,LC08,2013-09-18,2013,9,261,9,52,32.34,70.99,T1,L2SP,LANDSAT_8
9,LC08_L2SP_009052_20131020_20200912_02_T1,LC08,2013-10-20,2013,10,293,9,52,10.65,23.25,T1,L2SP,LANDSAT_8


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
362,LC09_L2SP_009052_20251216_20251217_02_T1,LC09,2025-12-16,2025,12,350,9,52,9.06,14.30,T1,L2SP,LANDSAT_9
363,LC08_L2SP_009052_20251224_20260106_02_T1,LC08,2025-12-24,2025,12,358,9,52,4.41,6.95,T1,L2SP,LANDSAT_8
364,LC09_L2SP_009052_20260101_20260102_02_T1,LC09,2026-01-01,2026,1,1,9,52,10.18,16.62,T1,L2SP,LANDSAT_9
365,LC08_L2SP_009052_20260109_20260115_02_T1,LC08,2026-01-09,2026,1,9,9,52,7.29,11.93,T1,L2SP,LANDSAT_8
366,LC09_L2SP_009052_20260117_20260118_02_T1,LC09,2026-01-17,2026,1,17,9,52,8.86,11.43,T1,L2SP,LANDSAT_9
367,LC08_L2SP_009052_20260125_20260204_02_T1,LC08,2026-01-25,2026,1,25,9,52,18.58,26.08,T1,L2SP,LANDSAT_8
368,LC08_L2SP_009052_20260210_20260220_02_T1,LC08,2026-02-10,2026,2,41,9,52,7.18,15.84,T1,L2SP,LANDSAT_8
369,LC09_L2SP_009052_20260218_20260219_02_T1,LC09,2026-02-18,2026,2,49,9,52,8.57,19.86,T1,L2SP,LANDSAT_9
370,LC08_L2SP_009052_20260226_20260306_02_T1,LC08,2026-02-26,2026,2,57,9,52,9.89,25.92,T1,L2SP,LANDSAT_8
371,LC09_L2SP_009052_20260306_20260307_02_T1,LC09,2026-03-06,2026,3,65,9,52,10.78,28.78,T1,L2SP,LANDSAT_9


In [7]:
# ============================================================
# CELDA 7 — Resumen anual
# ============================================================

annual = (
    df.groupby('year')
    .agg(
        n_scenes=('scene_id', 'count'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
        mean_cloud_cover=('cloud_cover', 'mean'),
        median_cloud_cover=('cloud_cover', 'median'),
        min_cloud_cover=('cloud_cover', 'min'),
        max_cloud_cover=('cloud_cover', 'max'),
        months=('month', lambda x: sorted(list(set(x)))),
    )
    .reset_index()
)

annual.to_csv(OUT_YEAR_CSV, index=False)

print('Resumen anual guardado en:')
print(OUT_YEAR_CSV)

display(annual)


Resumen anual guardado en:
/content/data/scene_inventory/annual_scene_count_landsat89.csv


,year,n_scenes,first_date,last_date,mean_cloud_cover,median_cloud_cover,min_cloud_cover,max_cloud_cover,months
0,2013,14,2013-04-01,2013-12-23,25.276429,16.700,2.82,73.55,"[4, 5, 6, 7, 8, 9, 10, 11, 12]"
1,2014,21,2014-01-24,2014-12-26,25.329524,15.210,1.23,89.76,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
2,2015,23,2015-01-11,2015-12-29,27.052174,17.740,2.22,80.98,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
3,2016,23,2016-01-14,2016-12-31,21.132174,14.450,0.89,97.70,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
4,2017,21,2017-01-16,2017-12-18,42.691429,20.470,4.36,100.00,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
5,2018,22,2018-01-03,2018-12-21,36.477727,27.555,0.33,100.00,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
6,2019,22,2019-01-06,2019-12-24,29.209545,23.190,4.48,94.47,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
7,2020,22,2020-01-09,2020-12-26,22.065455,19.060,2.51,73.20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
8,2021,25,2021-01-11,2021-12-29,30.426400,19.950,1.72,90.05,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
9,2022,42,2022-01-06,2022-12-24,31.824762,24.600,2.21,100.00,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"


In [8]:
# ============================================================
# CELDA 8 — Resumen mensual
# ============================================================

monthly = (
    df.groupby(['year', 'month'])
    .agg(
        n_scenes=('scene_id', 'count'),
        mean_cloud_cover=('cloud_cover', 'mean'),
        median_cloud_cover=('cloud_cover', 'median'),
        scene_ids=('scene_id', lambda x: ' | '.join(x)),
    )
    .reset_index()
    .sort_values(['year', 'month'])
)

monthly.to_csv(OUT_MONTH_CSV, index=False)

print('Resumen mensual guardado en:')
print(OUT_MONTH_CSV)

display(monthly)


Resumen mensual guardado en:
/content/data/scene_inventory/monthly_scene_count_landsat89.csv


,year,month,n_scenes,mean_cloud_cover,median_cloud_cover,scene_ids
0,2013,4,1,8.200000,8.200,LC08_L2SP_009052_20130401_20200913_02_T1
1,2013,5,1,16.200000,16.200,LC08_L2SP_009052_20130513_20200912_02_T1
2,2013,6,2,40.725000,40.725,LC08_L2SP_009052_20130614_20200912_02_T1 | LC0...
3,2013,7,1,73.550000,73.550,LC08_L2SP_009052_20130716_20200912_02_T1
4,2013,8,2,39.670000,39.670,LC08_L2SP_009052_20130801_20200912_02_T1 | LC0...
...,...,...,...,...,...,...
153,2026,1,4,11.227500,9.520,LC09_L2SP_009052_20260101_20260102_02_T1 | LC0...
154,2026,2,3,8.546667,8.570,LC08_L2SP_009052_20260210_20260220_02_T1 | LC0...
155,2026,3,4,16.025000,17.070,LC09_L2SP_009052_20260306_20260307_02_T1 | LC0...
156,2026,4,3,36.360000,38.900,LC09_L2SP_009052_20260407_20260408_02_T1 | LC0...


In [9]:
# ============================================================
# CELDA 9 — Verificación final de salidas
# ============================================================

expected_outputs = [OUT_SCENE_CSV, OUT_YEAR_CSV, OUT_MONTH_CSV]

for path in expected_outputs:
    if not path.exists():
        raise FileNotFoundError(f'No se generó el archivo esperado: {path}')
    if path.stat().st_size == 0:
        raise RuntimeError(f'Archivo generado vacío: {path}')

print('\n==============================')
print('MÓDULO 01 COMPLETADO')
print('==============================')

print('\nResumen anual rápido:')
for _, r in annual.iterrows():
    print(
        f"{int(r['year'])}: "
        f"{int(r['n_scenes'])} escenas | "
        f"meses={r['months']} | "
        f"cloud median={r['median_cloud_cover']:.1f}%"
    )

print('\nArchivos generados:')
for path in expected_outputs:
    print(path)



MÓDULO 01 COMPLETADO

Resumen anual rápido:
2013: 14 escenas | meses=[4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=16.7%
2014: 21 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=15.2%
2015: 23 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=17.7%
2016: 23 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=14.4%
2017: 21 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=20.5%
2018: 22 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=27.6%
2019: 22 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=23.2%
2020: 22 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=19.1%
2021: 25 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=19.9%
2022: 42 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=24.6%
2023: 43 escenas | meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | cloud median=14.4%
2024: 43 escenas |